<a href="https://colab.research.google.com/github/tabrejansary/ML-Assignments/blob/main/Lab-05/Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import os
import glob
import numpy as np
import pandas as pd

from PIL import Image
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DRIVE_DIR = "/content/drive/MyDrive"

DATASET_DIR = os.path.join(
    DRIVE_DIR,
    "BreaKHis_v1"
)

CSV_PATH = os.path.join(
    DRIVE_DIR,
    "BreaKHis_metadata.csv"
)

SAMPLES_PER_CLASS = 100

IMAGE_SIZE = (16, 16)

K = 3

DISTANCE_METRIC = "euclidean"

SORT_ALGORITHM = "insertion"

RANDOM_STATE = 42

In [ ]:
data = pd.read_csv(CSV_PATH)

print("Loaded:", CSV_PATH)

print("Columns:", data.columns.tolist())

print(data.head())

Loaded: /content/drive/MyDrive/BreaKHis_metadata.csv
Columns: ['fold', 'mag', 'grp', 'filename']
   fold  mag    grp                                           filename
0     1  100  train  BreaKHis_v1/histology_slides/breast/benign/SOB...
1     1  100  train  BreaKHis_v1/histology_slides/breast/benign/SOB...
2     1  100  train  BreaKHis_v1/histology_slides/breast/benign/SOB...
3     1  100  train  BreaKHis_v1/histology_slides/breast/benign/SOB...
4     1  100  train  BreaKHis_v1/histology_slides/breast/benign/SOB...


In [ ]:
# A1

def resolve_image_path(filename):
    filename = str(filename).replace("\\", "/")

    path1 = os.path.join(DRIVE_DIR, filename)

    if os.path.exists(path1):
        return path1

    path2 = os.path.join(DATASET_DIR, filename)

    if os.path.exists(path2):
        return path2

    if filename.startswith("BreaKHis_v1/"):
        filename = filename[len("BreaKHis_v1/"):]

        path3 = os.path.join(DATASET_DIR, filename)

        if os.path.exists(path3):
            return path3

    return None


def extract_class(filename):
    filename = str(filename).lower()

    if "/benign/" in filename:
        return "benign"

    if "/malignant/" in filename:
        return "malignant"

    return np.nan


def extract_image_features(image_path, image_size=(16, 16)):
    image = Image.open(image_path).convert("L")
    image = image.resize(image_size)

    image_array = np.asarray(image, dtype=float)

    features = image_array.flatten()

    return features


def prepare_image_dataset(data, samples_per_class=100):
    data = data.copy()

    data["class"] = data["filename"].apply(extract_class)

    data = data.dropna(subset=["class"])

    sampled_data = (
        data.groupby("class", group_keys=False)
        .apply(
            lambda group: group.sample(
                n=min(samples_per_class, len(group)),
                random_state=RANDOM_STATE
            )
        )
        .reset_index(drop=True)
    )

    feature_rows = []
    labels = []

    for _, row in sampled_data.iterrows():

        image_path = resolve_image_path(row["filename"])

        if image_path is None:
            continue

        try:
            features = extract_image_features(
                image_path,
                IMAGE_SIZE
            )

            feature_rows.append(features)
            labels.append(row["class"])

        except Exception:
            continue

    X = pd.DataFrame(feature_rows)
    y = pd.Series(labels, name="class")

    return X, y


def encode_data(data):
    encoded_data = data.copy()

    categorical_columns = encoded_data.select_dtypes(
        include=["object", "category"]
    ).columns

    for column in categorical_columns:

        categories = encoded_data[column].dropna().unique()

        mapping = {
            value: index
            for index, value in enumerate(categories)
        }

        encoded_data[column] = encoded_data[column].map(mapping)

    return encoded_data


def impute_missing_values(data, method="mean"):
    imputed_data = data.copy()

    for column in imputed_data.columns:

        if imputed_data[column].isnull().any():

            if method == "mean":
                value = imputed_data[column].mean()

            elif method == "median":
                value = imputed_data[column].median()

            elif method == "mode":
                value = imputed_data[column].mode()[0]

            else:
                raise ValueError(
                    "Method must be mean, median, or mode."
                )

            imputed_data[column] = (
                imputed_data[column].fillna(value)
            )

    return imputed_data


def calculate_distance(point1, point2, metric="euclidean"):
    point1 = np.asarray(point1, dtype=float)
    point2 = np.asarray(point2, dtype=float)

    if metric == "euclidean":

        return np.sqrt(
            np.sum((point1 - point2) ** 2)
        )

    elif metric == "manhattan":

        return np.sum(
            np.abs(point1 - point2)
        )

    else:

        raise ValueError(
            "Metric must be euclidean or manhattan."
        )


def bubble_sort(items):
    result = items.copy()

    n = len(result)

    for i in range(n):

        for j in range(0, n - i - 1):

            key1 = (result[j][0], result[j][2])
            key2 = (result[j + 1][0], result[j + 1][2])

            if key1 > key2:

                result[j], result[j + 1] = (
                    result[j + 1],
                    result[j]
                )

    return result


def selection_sort(items):
    result = items.copy()

    n = len(result)

    for i in range(n):

        minimum_index = i

        for j in range(i + 1, n):

            key1 = (
                result[j][0],
                result[j][2]
            )

            key2 = (
                result[minimum_index][0],
                result[minimum_index][2]
            )

            if key1 < key2:
                minimum_index = j

        result[i], result[minimum_index] = (
            result[minimum_index],
            result[i]
        )

    return result


def insertion_sort(items):
    result = items.copy()

    for i in range(1, len(result)):

        current = result[i]

        j = i - 1

        while j >= 0:

            previous_key = (
                result[j][0],
                result[j][2]
            )

            current_key = (
                current[0],
                current[2]
            )

            if previous_key <= current_key:
                break

            result[j + 1] = result[j]

            j -= 1

        result[j + 1] = current

    return result


def sort_distances(items, algorithm="bubble"):

    if algorithm == "bubble":
        return bubble_sort(items)

    elif algorithm == "selection":
        return selection_sort(items)

    elif algorithm == "insertion":
        return insertion_sort(items)

    else:
        raise ValueError(
            "Use bubble, selection, or insertion."
        )


def find_neighbors(sorted_distances, k):

    if k <= 0:
        raise ValueError("k must be greater than zero.")

    return sorted_distances[:k]


def assign_class(neighbors):

    class_counts = Counter()

    for distance, class_label, index in neighbors:
        class_counts[class_label] += 1

    maximum_votes = max(class_counts.values())

    tied_classes = [
        label
        for label, count in class_counts.items()
        if count == maximum_votes
    ]

    if len(tied_classes) == 1:
        return tied_classes[0]

    for distance, class_label, index in neighbors:

        if class_label in tied_classes:
            return class_label


def custom_knn_predict(
    X_train,
    y_train,
    X_test,
    k=3,
    metric="euclidean",
    algorithm="insertion"
):

    predictions = []

    for test_point in X_test:

        distance_list = []

        for index in range(len(X_train)):

            distance = calculate_distance(
                test_point,
                X_train[index],
                metric
            )

            distance_list.append(
                (
                    distance,
                    y_train[index],
                    index
                )
            )

        sorted_distances = sort_distances(
            distance_list,
            algorithm
        )

        neighbors = find_neighbors(
            sorted_distances,
            k
        )

        prediction = assign_class(neighbors)

        predictions.append(prediction)

    return np.array(predictions)

In [ ]:
#A2

def weighted_class_assignment(neighbors):

    class_weights = {}

    for distance, class_label, index in neighbors:

        weight = 1 / (distance + 1e-10)

        class_weights[class_label] = (
            class_weights.get(class_label, 0) + weight
        )

    maximum_weight = max(class_weights.values())

    tied_classes = [
        label
        for label, weight in class_weights.items()
        if weight == maximum_weight
    ]

    if len(tied_classes) == 1:
        return tied_classes[0]

    for distance, class_label, index in neighbors:

        if class_label in tied_classes:
            return class_label


def weighted_knn_predict(X_train, y_train, X_test,
                         k=3,
                         metric="euclidean",
                         algorithm="insertion"):

    predictions = []

    for test_point in X_test:

        distance_list = []

        for index in range(len(X_train)):

            distance = calculate_distance(
                test_point,
                X_train[index],
                metric
            )

            distance_list.append(
                (
                    distance,
                    y_train[index],
                    index
                )
            )

        sorted_distances = sort_distances(
            distance_list,
            algorithm
        )

        neighbors = find_neighbors(
            sorted_distances,
            k
        )

        prediction = weighted_class_assignment(
            neighbors
        )

        predictions.append(prediction)

    return np.array(predictions)

In [ ]:
#A3

def split_dataset(X, y, test_size=0.2,
                  random_state=42):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    return (
        X_train,
        X_test,
        y_train,
        y_test
    )


In [ ]:
#A4

def train_sklearn_knn(X_train, y_train, k=3):

    model = KNeighborsClassifier(
        n_neighbors=k
    )

    model.fit(
        X_train,
        y_train
    )

    return model

In [ ]:
#A5

def calculate_accuracy(y_actual, y_predicted):

    return accuracy_score(
        y_actual,
        y_predicted
    )

In [ ]:
#A6

def get_predictions(model, X_test):

    predictions = model.predict(
        X_test
    )

    return predictions

def prediction_table(y_actual, y_predicted):

    result = pd.DataFrame({
        "Actual": y_actual,
        "Predicted": y_predicted
    })

    result["Correct"] = (
        result["Actual"] == result["Predicted"]
    )

    return result


In [ ]:
#A7

class CustomKNN:

    def __init__(self, k=3,
                 metric="euclidean",
                 algorithm="insertion"):

        self.k = k
        self.metric = metric
        self.algorithm = algorithm

        self.X_train = None
        self.y_train = None


    def fit(self, X_train, y_train):

        self.X_train = np.asarray(
            X_train,
            dtype=float
        )

        self.y_train = np.asarray(
            y_train
        )

        return self


    def predict(self, X_test):

        X_test = np.asarray(
            X_test,
            dtype=float
        )

        return custom_knn_predict(
            self.X_train,
            self.y_train,
            X_test,
            self.k,
            self.metric,
            self.algorithm
        )


    def score(self, X_test, y_test):

        predictions = self.predict(
            X_test
        )

        return accuracy_score(
            y_test,
            predictions
        )

In [ ]:
#A8

def compare_k_values(X_train, y_train,
                     X_test, y_test,
                     k_values,
                     metric="euclidean",
                     algorithm="insertion"):

    results = []

    for k in k_values:

        custom_prediction = custom_knn_predict(
            X_train,
            y_train,
            X_test,
            k,
            metric,
            algorithm
        )

        sklearn_model = train_sklearn_knn(
            X_train,
            y_train,
            k
        )

        sklearn_prediction = sklearn_model.predict(
            X_test
        )

        custom_accuracy = accuracy_score(
            y_test,
            custom_prediction
        )

        sklearn_accuracy = accuracy_score(
            y_test,
            sklearn_prediction
        )

        results.append({
            "k": k,
            "Custom kNN Accuracy": custom_accuracy,
            "Sklearn kNN Accuracy": sklearn_accuracy
        })

    return pd.DataFrame(results)

In [ ]:
#A9

def compare_weighted_knn(X_train, y_train,
                         X_test, y_test,
                         k_values,
                         metric="euclidean",
                         algorithm="insertion"):

    results = []

    for k in k_values:

        normal_prediction = custom_knn_predict(
            X_train,
            y_train,
            X_test,
            k,
            metric,
            algorithm
        )

        weighted_prediction = weighted_knn_predict(
            X_train,
            y_train,
            X_test,
            k,
            metric,
            algorithm
        )

        normal_accuracy = accuracy_score(
            y_test,
            normal_prediction
        )

        weighted_accuracy = accuracy_score(
            y_test,
            weighted_prediction
        )

        results.append({
            "k": k,
            "Normal kNN Accuracy": normal_accuracy,
            "Weighted kNN Accuracy": weighted_accuracy
        })

    return pd.DataFrame(results)

In [ ]:
#main

data = pd.read_csv(CSV_PATH)

X, y = prepare_image_dataset(
    data,
    samples_per_class=SAMPLES_PER_CLASS
)

X = impute_missing_values(
    X,
    method="mean"
)

X = encode_data(X)

X = X.to_numpy(
    dtype=float
)

y = y.to_numpy()


print("Dataset Shape:", X.shape)
print("\nClass Distribution:")
print(pd.Series(y).value_counts())


#A1

test_index = 0

X_demo_train = np.delete(
    X,
    test_index,
    axis=0
)

y_demo_train = np.delete(
    y,
    test_index
)

X_demo_test = X[
    test_index:test_index + 1
]

actual_class = y[
    test_index
]

prediction = custom_knn_predict(
    X_demo_train,
    y_demo_train,
    X_demo_test,
    k=K,
    metric=DISTANCE_METRIC,
    algorithm=SORT_ALGORITHM
)

print("\nA1")
print("Actual Class:", actual_class)
print("Predicted Class:", prediction[0])


#A2

weighted_prediction = weighted_knn_predict(
    X_demo_train,
    y_demo_train,
    X_demo_test,
    k=K,
    metric=DISTANCE_METRIC,
    algorithm=SORT_ALGORITHM
)

print("\nA2")
print("Actual Class:", actual_class)
print("Weighted kNN Prediction:", weighted_prediction[0])


#A3

X_train, X_test, y_train, y_test = split_dataset(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("\nA3")
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))


#A4

sklearn_model = train_sklearn_knn(
    X_train,
    y_train,
    k=3
)

sklearn_predictions = get_predictions(
    sklearn_model,
    X_test
)

print("\nA4")
print("Scikit-learn kNN trained with k =", 3)


#A5

sklearn_accuracy = calculate_accuracy(
    y_test,
    sklearn_predictions
)

print("\nA5")
print("Scikit-learn Accuracy:",
      round(sklearn_accuracy, 4))


#A6

results_A6 = prediction_table(
    y_test,
    sklearn_predictions
)

print("\nA6")
display(results_A6.head(10))


#A7

custom_model = CustomKNN(
    k=3,
    metric=DISTANCE_METRIC,
    algorithm=SORT_ALGORITHM
)

custom_model.fit(
    X_train,
    y_train
)

custom_predictions = custom_model.predict(
    X_test
)

custom_accuracy = custom_model.score(
    X_test,
    y_test
)

print("\nA7")
print("Custom kNN Accuracy:",
      round(custom_accuracy, 4))


#A8

k_values = [1, 3, 5, 7, 9]

results_A8 = compare_k_values(
    X_train,
    y_train,
    X_test,
    y_test,
    k_values,
    metric=DISTANCE_METRIC,
    algorithm=SORT_ALGORITHM
)

print("\nA8")
display(results_A8)


#A9

results_A9 = compare_weighted_knn(
    X_train,
    y_train,
    X_test,
    y_test,
    k_values,
    metric=DISTANCE_METRIC,
    algorithm=SORT_ALGORITHM
)

print("\nA9")
display(results_A9)

/tmp/ipykernel_590/3807125097.py:59: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Dataset Shape: (200, 256)

Class Distribution:
benign       100
malignant    100
Name: count, dtype: int64

A1
Actual Class: benign
Predicted Class: benign

A2
Actual Class: benign
Weighted kNN Prediction: benign

A3
Training Samples: 160
Testing Samples: 40

A4
Scikit-learn kNN trained with k = 3

A5
Scikit-learn Accuracy: 0.525

A6


,Actual,Predicted,Correct
0,benign,benign,True
1,malignant,malignant,True
2,malignant,malignant,True
3,malignant,malignant,True
4,benign,malignant,False
5,malignant,benign,False
6,benign,benign,True
7,malignant,malignant,True
8,malignant,malignant,True
9,benign,malignant,False



A7
Custom kNN Accuracy: 0.525

A8


,k,Custom kNN Accuracy,Sklearn kNN Accuracy
0,1,0.525,0.525
1,3,0.525,0.525
2,5,0.475,0.475
3,7,0.525,0.525
4,9,0.550,0.550



A9


,k,Normal kNN Accuracy,Weighted kNN Accuracy
0,1,0.525,0.525
1,3,0.525,0.525
2,5,0.475,0.475
3,7,0.525,0.525
4,9,0.550,0.550
